# EDA 02 · Calidad, identificadores y consistencia mensual

**Proyecto:** educación superior y empleo en Ecuador, ENEMDU 2025.

**Objetivo de aprendizaje:** distinguir un dato inválido, una ausencia esperada y una repetición de encuesta. Este cuaderno no elimina filas ni cambia los archivos originales.

Trabajaremos con los reportes reales producidos por `src/01_auditoria_calidad.py`, cuyo código incluye comentarios de etapa, bloque y línea. La auditoría perfila todas las columnas y aplica reglas semánticas a las variables del estudio; no certifica cada respuesta ni reconstruye todos los saltos ocupacionales.

Ejecuta las celdas de arriba hacia abajo. Si un concepto es nuevo, detente en la pregunta de interpretación antes de continuar.

## Paso 1. Encontrar el proyecto

Una **ruta** indica dónde está un archivo. `Path` evita escribir una ruta personal fija. Buscamos la carpeta del proyecto partiendo de la ubicación donde abriste Jupyter.

In [4]:
# BLOQUE 1: importar herramientas y localizar la carpeta del proyecto.
from pathlib import Path  # Maneja rutas de archivos.
import hashlib  # Verifica la identidad del archivo original.
import importlib.util  # Permite cargar nuestro programa comentado.
import json  # Lee los reportes estructurados.
import pandas as pd  # Trabaja con tablas.
from IPython.display import display, Markdown  # Presenta tablas y explicaciones.

ubicacion = Path.cwd().resolve()  # Obtiene la carpeta desde la que se ejecuta el cuaderno.
posibles = [ubicacion, *ubicacion.parents]  # Examina también las carpetas superiores.
raiz = next((p for p in posibles if (p / "src/01_auditoria_calidad.py").exists()), None)  # Busca el programa del proyecto.
if raiz is None:  # Contempla que Jupyter se abra en la carpeta que contiene enemdu_2025.
    raiz = ubicacion / "enemdu_2025"  # Prueba la ubicación habitual dentro del espacio de trabajo.
assert (raiz / "src/01_auditoria_calidad.py").exists(), "Abre este cuaderno dentro de la carpeta enemdu_2025."  # Verifica la ubicación.
reportes = raiz / "reports/calidad"  # Define dónde leeremos las salidas.
print("Proyecto localizado:", raiz.name)  # Confirma la localización sin depender de una ruta personal.

Proyecto localizado: Enemdu_2025


## Paso 2. Reproducir o consultar

La primera auditoría ya está ejecutada. Deja `RECALCULAR = False` para estudiar sus resultados. Cambia a `True` si deseas recorrer de nuevo las tablas completas; puede tardar varios minutos. **Recalcular no modifica los originales.**

In [2]:
# BLOQUE 2: ejecutar opcionalmente el programa de auditoría.
RECALCULAR = False  # Cambia a True únicamente si deseas repetir todos los controles.
if RECALCULAR or not (reportes / "00_resumen_auditoria.json").exists():  # Ejecuta también si faltan resultados.
    especificacion = importlib.util.spec_from_file_location("auditoria", raiz / "src/01_auditoria_calidad.py")  # Localiza el módulo.
    modulo = importlib.util.module_from_spec(especificacion)  # Prepara el módulo en memoria.
    especificacion.loader.exec_module(modulo)  # Carga las funciones sin alterar la base.
    modulo.auditar()  # Ejecuta los controles comentados del programa principal.
resumen = json.loads((reportes / "00_resumen_auditoria.json").read_text(encoding="utf-8"))  # Lee los resultados.
huella = hashlib.sha256((raiz / "data/raw/2_BDD_DATOS_ABIERTOS_ENEMDU_2025_CSV.zip").read_bytes()).hexdigest()  # Calcula la huella actual.
assert huella == resumen["sha256_original"], "El ZIP cambió: recalcula antes de interpretar estos reportes."  # Evita leer resultados de otra fuente.
display(pd.DataFrame({"Medida": ["Registros de personas", "Registros vivienda-hogar", "Variables de personas", "Reglas distintas"], "Valor": [resumen["filas_personas"], resumen["filas_vivienda_hogar"], resumen["columnas_personas"], resumen["reglas_distintas"]]}))  # Presenta el tamaño auditado.

,Medida,Valor
0,Registros de personas,334786
1,Registros vivienda-hogar,104738
2,Variables de personas,139
3,Reglas distintas,73


## Paso 3. Leer la cobertura mensual

Cada fila de esta tabla resume un mes. Los pesos proceden de la **base anual**: sumar los pesos de un mes no produce automáticamente la estimación mensual oficial. “Registros candidatos” tampoco significa personas distintas durante el año.

In [3]:
# BLOQUE 3: comprobar que los doce meses recuperan todos los registros.
mensual = pd.read_csv(reportes / "10_cobertura_mensual.csv", dtype={"mes": str})  # Conserva los ceros de los meses.
display(mensual[["mes", "registros", "hogares_observacion", "upm", "provincias", "candidatos", "edad_99"]])  # Muestra cobertura y población candidata.
assert mensual["registros"].sum() == resumen["filas_personas"]  # Comprueba una conciliación básica.
print("Los recuentos mensuales coinciden con el total anual de registros.")  # Explica el resultado.

,mes,registros,hogares_observacion,upm,provincias,candidatos,edad_99
0,01,28320,8795,1288,24,3286,0
1,02,28303,8766,1288,24,3174,0
2,03,27932,8790,1288,24,3263,0
3,04,28092,8810,1288,24,3297,0
4,05,27903,8739,1288,24,3243,0
5,06,28072,8768,1288,24,3318,0
6,07,27509,8607,1288,24,3289,0
7,08,27580,8607,1288,24,3314,0
8,09,27332,8584,1288,24,3306,0
9,10,27809,8757,1288,24,3343,0


Los recuentos mensuales coinciden con el total anual de registros.


## Paso 4. Interpretar faltantes con su denominador

Un **vacío físico** es una celda sin contenido. Puede corresponder a una pregunta que no se aplica. Una **no respuesta codificada** contiene un número reservado, como 999999 en determinados ingresos. No se deben tratar igual que un cero real.

Las reglas muestran cuántos registros debían cumplir cada condición (`n_aplicables`) y cuántos requieren revisión (`n_alertas`). Una alerta no demuestra un error.

In [4]:
# BLOQUE 4: consultar controles y faltantes del dominio de estudio.
reglas = pd.read_csv(reportes / "03_reglas_calidad.csv", dtype={"mes": str})  # Lee todas las reglas con sus denominadores.
anuales = reglas.loc[reglas["mes"].eq("ANUAL")]  # Selecciona el resumen anual.
display(anuales.loc[anuales["n_alertas"].gt(0), ["regla", "descripcion", "tipo", "n_aplicables", "n_alertas"]])  # Muestra señales que requieren interpretación.
display(anuales.loc[anuales["tipo"].eq("faltante_candidato"), ["descripcion", "n_aplicables", "n_alertas"]])  # Examina covariables dentro de la población candidata.

,regla,descripcion,tipo,n_aplicables,n_alertas
26,Q04,Edad 98: categoría abierta 98 y más según etiq...,codigo_especial,334786,224
624,NR_p63,No respuesta 999999 en p63,codigo_especial,334786,239
637,NR_p64b,No respuesta 999999 en p64b,codigo_especial,334786,21
650,NR_p65,No respuesta 999999 en p65,codigo_especial,334786,234
663,NR_p66,No respuesta 999999 en p66,codigo_especial,334786,241
676,NR_p67,No respuesta 999999 en p67,codigo_especial,334786,231
689,NR_p68b,No respuesta 999999 en p68b,codigo_especial,334786,37
702,NR_p69,No respuesta 999999 en p69,codigo_especial,334786,10
715,NR_p70b,No respuesta 999999 en p70b,codigo_especial,334786,1
728,NR_p71b,No respuesta 999999 en p71b,codigo_especial,334786,23


,descripcion,n_aplicables,n_alertas
507,p02 vacío en población candidata,39551,0
520,p06 vacío en población candidata,39551,0
533,p07 vacío en población candidata,39551,0
546,p10a vacío en población candidata,39551,0
559,p12a vacío en población candidata,39551,0
572,p15 vacío en población candidata,39551,0
585,prov vacío en población candidata,39551,0
598,area vacío en población candidata,39551,0
611,p10b vacío en población candidata,39551,0


**Pregunta de interpretación:** ¿por qué sería incorrecto eliminar toda fila que tenga al menos una celda vacía? Busca una pregunta que solo corresponda a quienes tienen trabajo o a quienes estudiaron educación superior.

In [5]:
# BLOQUE 5: explorar disponibilidad de variables sin limpiar los datos.
perfil = pd.read_csv(reportes / "01_perfil_variables_mes.csv", dtype={"mes": str})  # Lee el perfil de todas las columnas.
seleccion = perfil["tabla"].eq("personas") & perfil["mes"].eq("ANUAL") & perfil["variable"].isin(["p03", "p10a", "p12a", "p12b", "ingrl"])  # Escoge ejemplos del estudio.
display(perfil.loc[seleccion, ["variable", "n", "vacios", "pct_vacios", "distintos_no_vacios"]])  # Compara ausencia física sin atribuirle causas automáticamente.
cambios = pd.read_csv(reportes / "02_cambios_disponibilidad.csv")  # Lee cambios entre meses.
display(cambios.head(10))  # Identifica las mayores variaciones para revisar módulos y saltos.

,variable,n,vacios,pct_vacios,distintos_no_vacios
8,p03,334786,0,0.000000,99
16,p10a,334786,17399,5.197051,9
19,p12a,334786,263508,78.709385,2
20,p12b,334786,286507,85.579146,1052
121,ingrl,334786,188650,56.349429,2932


,tabla,variable,pct_vacio_min,pct_vacio_max,meses_totalmente_vacios,diferencia_pp
0,personas,p26,69.249856,74.795198,0,5.545342
1,personas,p25,76.396803,81.810990,0,5.414186
2,personas,p22,51.320621,53.901378,0,2.580756
3,personas,p15ab,62.304883,64.648368,0,2.343485
4,personas,p21,49.851695,52.103711,0,2.252016
5,personas,p73b,76.235735,78.316372,0,2.080638
6,personas,p28,89.565678,91.473677,0,1.907999
7,personas,p30,89.826977,91.707422,0,1.880445
8,personas,p29,89.826977,91.707422,0,1.880445
9,personas,ingrl,55.514866,57.334040,0,1.819174


## Paso 5. Separar unicidad de seguimiento

Una clave puede ser única porque incorpora el mes. Eso no garantiza que una vivienda no reaparezca en otro levantamiento. Retiramos el sufijo mensual **solo después de comprobar su composición en todas las filas**.

La clave de persona sin mes representa una posición dentro del hogar. Cambios de miembros, numeración o respuesta pueden impedir que corresponda siempre a la misma persona física.

In [6]:
# BLOQUE 6: examinar duplicados y repeticiones temporales.
duplicados = pd.read_csv(reportes / "04_duplicados.csv")  # Consulta unicidad de claves y filas completas.
repeticiones = pd.read_csv(reportes / "09_repeticion_sin_mes.csv")  # Consulta repeticiones una vez retirado el mes validado.
display(duplicados)  # Distingue repeticiones exactas de coincidencias de identificador.
display(repeticiones)  # Muestra cuántas claves estructurales aparecen en varios meses.
display(pd.read_csv(reportes / "08_cambios_claves_persona.csv"))  # Muestra señales que limitan el seguimiento individual.

,tabla,clave,filas,claves_distintas,filas_en_claves_repetidas,repeticiones_adicionales
0,personas,id_persona,334786,334786,0,0
1,personas_mes,id_persona+mes,334786,334786,0,0
2,vivienda_hogar,id_hogar,104738,104738,0,0
3,vivienda_hogar_mes,id_hogar+mes,104738,104738,0,0
4,personas,TODAS_LAS_COLUMNAS,334786,334786,0,0
5,vivienda_hogar,TODAS_LAS_COLUMNAS,104738,104738,0,0


,nivel,claves_sin_mes,claves_en_varios_meses,max_meses,filas_en_claves_repetidas
0,id_vivienda,67117,37216,2,240391
1,id_hogar,67408,37330,2,240115
2,id_persona,219778,115008,2,230016


,revision,pares_comparables,pares_con_senal
0,sexo_cambia,115008,5969
1,edad_disminuye_o_aumenta_mas_de_un_ano,114913,25871
2,titulo_si_a_no,115008,653


**Pregunta de interpretación:** si repartimos las filas al azar, ¿podrían dos visitas a la misma vivienda terminar en entrenamiento y prueba? Explica por qué una separación por grupos reduce ese riesgo.

In [7]:
# BLOQUE 7: consultar la matriz de solapamiento mensual de hogares.
solapamiento = pd.read_csv(reportes / "07_solapamiento_id_hogar.csv", dtype={"mes_a": str, "mes_b": str})  # Lee claves compartidas entre meses.
matriz = solapamiento.pivot(index="mes_a", columns="mes_b", values="claves_compartidas")  # Organiza meses en filas y columnas.
display(matriz)  # Presenta los recuentos sin añadir dependencias de gráficos en esta etapa.

mes_b,01,02,03,04,05,06,07,08,09,10,11,12
mes_a,,,,,,,,,,,,
01,8795,0,0,3693,0,0,0,0,0,1980,0,0
02,0,8766,0,0,3657,0,0,0,0,0,1944,0
03,0,0,8790,0,0,3611,0,0,0,0,0,1973
04,3693,0,0,8810,0,0,3661,0,0,0,0,0
05,0,3657,0,0,8739,0,0,3599,0,0,0,0
06,0,0,3611,0,0,8768,0,0,3626,0,0,0
07,0,0,0,3661,0,0,8607,0,0,3193,0,0
08,0,0,0,0,3599,0,0,8607,0,0,3228,0
09,0,0,0,0,0,3626,0,0,8584,0,0,3165


La diagonal compara cada mes consigo mismo. Las celdas fuera de la diagonal muestran revisitas posibles. Esta matriz no demuestra por sí sola que cada integrante del hogar sea la misma persona.

## Paso 6. Verificar el enlace entre tablas y el diseño

La unión personas–hogares debe ser **muchos a uno**: muchas personas pueden pertenecer a un hogar, pero una observación de persona no debe multiplicarse al unir tablas. Los estratos y conglomerados se conservan en la muestra completa para la futura inferencia de dominios.

In [8]:
# BLOQUE 8: revisar integridad y número de conglomerados.
display(pd.read_csv(reportes / "06_integridad_personas_hogares.csv"))  # Muestra registros sin pareja y discrepancias de campos comunes.
display(pd.read_csv(reportes / "05_dependencias_identificadores.csv"))  # Comprueba la jerarquía de identificadores.
diseno = pd.read_csv(reportes / "11_upm_por_estrato.csv", dtype={"estrato": str})  # Lee UPM de muestra completa y dominio.
display(diseno.loc[diseno["upm_muestra_completa"].le(1) | diseno["upm_con_candidatos"].le(1)])  # Muestra estratos con poca presencia para revisar incertidumbre.
print("Estratos de la muestra completa:", len(diseno))  # Informa cobertura del diseño.

,control,n
0,personas_sin_hogar_en_tabla_vivienda,0
1,hogares_tabla_vivienda_sin_personas,0
2,diferencia_area_persona_vivienda,0
3,diferencia_upm_persona_vivienda,0
4,diferencia_estrato_persona_vivienda,0
5,diferencia_fexp_persona_vivienda,0


,clave,atributo,n_claves,claves_con_mas_de_un_valor
0,id_persona,id_hogar,334786,0
1,id_persona,id_vivienda,334786,0
2,id_persona,upm,334786,0
3,id_persona,mes,334786,0
4,id_hogar,id_vivienda,104738,0
5,id_hogar,upm,104738,0
6,id_hogar,mes,104738,0
7,id_hogar,estrato,104738,0
8,id_hogar,prov,104738,0
9,id_hogar,area,104738,0


,estrato,upm_muestra_completa,upm_con_candidatos


Estratos de la muestra completa: 150


## Paso 7. Comprobar la consistencia de los formularios

Comparamos las preguntas 10 y 12 de los doce PDF oficiales. El texto se normaliza quitando diferencias de espacios y acentos. La coincidencia respalda el mismo significado de estas preguntas, pero no demuestra que todos los módulos del cuestionario sean idénticos.

In [9]:
# BLOQUE 9: consultar evidencia documental mensual.
formularios = json.loads((reportes / "formularios_mensuales.json").read_text(encoding="utf-8"))  # Lee páginas, enlaces y huellas.
display(pd.DataFrame([{"mes": f["mes"], "estado": f["estado"], "pagina_nivel": f.get("pregunta10_pagina_pdf"), "titulo_si1_no2": f.get("pregunta12_si1_no2", False)} for f in formularios]))  # Presenta la cobertura documental.
print("Textos distintos de pregunta 10:", len({f.get("pregunta10_sha256_texto") for f in formularios}))  # Resume equivalencia del texto educativo.
print("Fragmentos distintos de pregunta 12:", len({f.get("pregunta12_sha256_fragmento") for f in formularios}))  # Resume equivalencia de titulación.

,mes,estado,pagina_nivel,titulo_si1_no2
0,01,pregunta_localizada,3,True
1,02,pregunta_localizada,3,True
2,03,pregunta_localizada,3,True
3,04,pregunta_localizada,3,True
4,05,pregunta_localizada,3,True
5,06,pregunta_localizada,3,True
6,07,pregunta_localizada,3,True
7,08,pregunta_localizada,3,True
8,09,pregunta_localizada,3,True
9,10,pregunta_localizada,3,True


Textos distintos de pregunta 10: 1
Fragmentos distintos de pregunta 12: 1


## Paso 8. Contrastar códigos con las etiquetas oficiales

El archivo SPSS del INEC conserva etiquetas de valores que no aparecen en el diccionario Excel. Verificamos 89 variables categóricas contra ese catálogo. En edad, 98 significa **98 y más** y 99 significa **no informa**. En ingreso laboral, -1 significa **gasta más de lo que gana** y 999999 significa **no informa**.

El valor -1 no mide una pérdida de exactamente un dólar. Tampoco debe confundirse con una respuesta faltante.

In [10]:
# BLOQUE 10: leer la validación de códigos y los estados del ingreso.
etiquetas = json.loads((reportes / "17_resumen_etiquetas.json").read_text(encoding="utf-8"))  # Lee el contraste con el SAV oficial.
print("Variables categóricas validadas:", etiquetas["variables_categoricas_validadas"])  # Informa el alcance de esta comprobación.
print("Categorías no etiquetadas encontradas:", len(etiquetas["alertas_anuales"]))  # No confunde ausencia de códigos inválidos con ausencia de cualquier problema.
display(pd.read_csv(reportes / "18_ingresos_por_condicion.csv"))  # Distingue no remuneración, no respuesta y gasto superior al ingreso.
print("Diferencias ciudad–prefijo UPM:", etiquetas["filas_prefijo_upm_distinto_ciudad"])  # Expone la revisión cartográfica pendiente.
print("Diferencias provincia–prefijo UPM:", etiquetas["filas_provincia_prefijo_upm_distinta_prov"])  # Comprueba consistencia territorial al nivel usado en el estudio.

Variables categóricas validadas: 89
Categorías no etiquetadas encontradas: 0


,condact,gasta_mas_de_lo_que_gana,no_informa,vacio,valor_no_negativo
0,1,0,0,0,27594
1,2,28,10,431,3450
2,3,0,0,0,285
3,4,118,0,0,3751
4,5,0,0,981,0
5,6,0,258,0,0


Diferencias ciudad–prefijo UPM: 1787
Diferencias provincia–prefijo UPM: 0


## Paso 9. Decidir antes de limpiar

Consulta `docs/04_informe_calidad.md` para la interpretación y las decisiones. En la siguiente etapa construiremos la base tratada, registrando cada cambio. Todavía no hemos imputado ingresos, eliminado observaciones repetidas ni seleccionado modelos por su rendimiento.

**Ejercicio:** explica con tus palabras (1) por qué una celda vacía puede ser correcta, (2) por qué un identificador único puede ocultar revisitas y (3) por qué 999999 no debe convertirse automáticamente en un ingreso alto.

**Fuentes:** [guía anual](https://www.ecuadorencifras.gob.ec/documentos/web-inec/EMPLEO/2025/anual/Guia_de_usuario_BDD_ENEMDU_anual_2025.pdf), [diseño anual](https://www.ecuadorencifras.gob.ec/documentos/web-inec/EMPLEO/2025/anual/Disenio_Muestral_ENEMDU_Anual_enero-diciembre_2025.pdf), diccionarios y sintaxis guardados en `data/raw/documentacion`. Los enlaces exactos de cada formulario están en `formularios_mensuales.json`.